# 57 RoBERTa LoRA Improvement Experiments

Use this notebook to run controlled RoBERTa LoRA variants based on notebook 50 without changing notebook 50 itself.


In [ ]:
# Cell 0: Configure 57c RoBERTa-large hyperparameter push (full-data final).
import os
import re
import ast
import json
import textwrap
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/rameyjm7/workspace/masked-emotion-lora-benchmark')
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
REGISTRY_PATH = PROJECT_ROOT / 'configs' / 'model_registry.yaml'
ENV_FILE = PROJECT_ROOT / 'env.txt'
ENV_PKL_FILE = PROJECT_ROOT / 'configs' / 'env.pkl'

MODEL_ID = 'roberta_large'

# Choose which experiment entry to run.
ACTIVE_EXPERIMENT = 5
EXPERIMENT_DEFAULTS = {
    'train_rows': None,
    'max_length': 512,
    'lr': 1.5e-5,
    'epochs': 4.0,
    'train_batch_size': 4,
    'grad_accum': 3,
    'lora_r': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.05,
    'target_modules': ['query', 'key', 'value', 'dense'],
    'seed': 42,
    'val_ratio': 0.12,
    'early_stopping_patience': 3,
    'early_stopping_threshold': 0.0,
    'weight_decay': 0.02,
    'warmup_ratio': 0.10,
    'save_total_limit': 2,
    'shuffle_train_rows': True,
}

EXPERIMENTS = [
    dict(EXPERIMENT_DEFAULTS, **{
        'name': '57c_full_qkv_dense_ref',
        'seed': 42,
    }),
    dict(EXPERIMENT_DEFAULTS, **{
        'name': '57c_full_qkv_dense_rank48',
        'lr': 1e-5,
        'epochs': 5.0,
        'lora_r': 48,
        'lora_alpha': 96,
        'lora_dropout': 0.03,
        'seed': 2026,
    }),
    dict(EXPERIMENT_DEFAULTS, **{
        'name': '57c_full_qkv_dense_rank64_lowdrop',
        'lr': 8e-6,
        'epochs': 6.0,
        'grad_accum': 4,
        'lora_r': 64,
        'lora_alpha': 128,
        'lora_dropout': 0.02,
        'weight_decay': 0.03,
        'warmup_ratio': 0.12,
        'seed': 314,
    }),
    dict(EXPERIMENT_DEFAULTS, **{
        'name': '57c_full_qkv_dense_rank64_seed1337',
        'lr': 8e-6,
        'epochs': 6.0,
        'grad_accum': 4,
        'lora_r': 64,
        'lora_alpha': 128,
        'lora_dropout': 0.02,
        'weight_decay': 0.03,
        'warmup_ratio': 0.12,
        'seed': 1337,
    }),
    dict(EXPERIMENT_DEFAULTS, **{
        'name': '57c_final_full_rank64_conservative',
        'lr': 6e-6,
        'epochs': 7.0,
        'grad_accum': 4,
        'lora_r': 64,
        'lora_alpha': 128,
        'lora_dropout': 0.02,
        'weight_decay': 0.03,
        'warmup_ratio': 0.12,
        'val_ratio': 0.10,
        'early_stopping_patience': 4,
        'save_total_limit': 3,
        'seed': 42,
    }),
    dict(EXPERIMENT_DEFAULTS, **{
        'name': '57c_final_full_rank64_noval_canonical',
        'train_rows': None,
        'lr': 8e-6,
        'epochs': 6.0,
        'grad_accum': 4,
        'lora_r': 64,
        'lora_alpha': 128,
        'lora_dropout': 0.02,
        'weight_decay': 0.03,
        'warmup_ratio': 0.12,
        'val_ratio': 0.0,
        'early_stopping_patience': 0,
        'save_total_limit': 1,
        'seed': 314,
    }),
]

assert 0 <= ACTIVE_EXPERIMENT < len(EXPERIMENTS), 'ACTIVE_EXPERIMENT out of range.'
EXP = EXPERIMENTS[ACTIVE_EXPERIMENT]
EXPERIMENT_NAME = EXP['name']
EXPERIMENT_MODEL_ID = f"{MODEL_ID}_{EXPERIMENT_NAME}"

# Distributed training controls.
REQUEST_DISTRIBUTED_TRAIN = True
MAX_NPROC_PER_NODE = 2
MASTER_PORT = 29651
GPU_IDS = '0,1'      # Requested GPUs when distributed mode is enabled.
SINGLE_GPU_ID = '0'  # Preferred single GPU id when distributed mode is disabled.

# Normalize requested GPU ids (dedupe while preserving order).
_gpu_id_list = [x.strip() for x in str(GPU_IDS).split(',') if x.strip()]
_gpu_id_list = list(dict.fromkeys(_gpu_id_list))
GPU_IDS = ','.join(_gpu_id_list)

if REQUEST_DISTRIBUTED_TRAIN and GPU_IDS:
    os.environ['CUDA_VISIBLE_DEVICES'] = GPU_IDS
elif SINGLE_GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(SINGLE_GPU_ID)

# Auto-scale mode based on actual visible GPUs in the current allocation.
try:
    import torch
    VISIBLE_GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    VISIBLE_GPU_COUNT = 0

DISTRIBUTED_TRAIN = bool(REQUEST_DISTRIBUTED_TRAIN and VISIBLE_GPU_COUNT >= 2)
NPROC_PER_NODE = min(MAX_NPROC_PER_NODE, VISIBLE_GPU_COUNT) if DISTRIBUTED_TRAIN else 1

# If we cannot run distributed, pin to a single visible GPU if specified.
if not DISTRIBUTED_TRAIN and SINGLE_GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(SINGLE_GPU_ID)

RUN_TRAIN = True
RUN_INFERENCE = True
RUN_EVAL = True

TRAIN_ROWS = EXP['train_rows']
MAX_LENGTH = int(EXP['max_length'])
LR = float(EXP['lr'])
EPOCHS = float(EXP['epochs'])
TRAIN_BATCH_SIZE = int(EXP['train_batch_size'])
GRAD_ACCUM = int(EXP['grad_accum'])
LORA_R = int(EXP['lora_r'])
LORA_ALPHA = int(EXP['lora_alpha'])
LORA_DROPOUT = float(EXP['lora_dropout'])
TARGET_MODULES = list(EXP['target_modules'])
SEED = int(EXP.get('seed', 42))
VAL_RATIO = float(EXP.get('val_ratio', 0.10))
EARLY_STOPPING_PATIENCE = int(EXP.get('early_stopping_patience', 2))
EARLY_STOPPING_THRESHOLD = float(EXP.get('early_stopping_threshold', 0.0))
WEIGHT_DECAY = float(EXP.get('weight_decay', 0.01))
WARMUP_RATIO = float(EXP.get('warmup_ratio', 0.06))
SAVE_TOTAL_LIMIT = int(EXP.get('save_total_limit', 2))
SHUFFLE_TRAIN_ROWS = bool(EXP.get('shuffle_train_rows', True))

# Queue controls: set True to run additional large-model experiments automatically.
RUN_EXPERIMENT_QUEUE = False
EXPERIMENT_QUEUE = []
SKIP_COMPLETED_IN_QUEUE = True

MODEL_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / f'hf_cache_{MODEL_ID}'
RUN_ROOT = PROJECT_ROOT / 'outputs' / 'lora_runs' / f'{MODEL_ID}_experiments'
RUN_DIR = RUN_ROOT / EXPERIMENT_NAME
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics' / f'{MODEL_ID}_experiments'
METRICS_DIR = METRICS_ROOT / EXPERIMENT_NAME
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / f'{MODEL_ID}_experiments'

EVAL_INPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_baseline_eval_input.csv'
RAW_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_lora_raw.csv'
CLEAN_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_lora_clean.csv'
TRAIN_PREP_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_train_prep_stats.json'
METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'
EXP_SUMMARY_PATH = METRICS_ROOT / 'experiment_summary.csv'
FIGURE_PATH = FIGURES_DIR / f'figure2_with_lora_overlay_{EXPERIMENT_MODEL_ID}.png'

for p in [MODEL_CACHE_DIR, RUN_ROOT, RUN_DIR, METRICS_ROOT, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('MODEL_ID =', MODEL_ID)
print('EXPERIMENT_NAME =', EXPERIMENT_NAME)
print('EXPERIMENT_MODEL_ID =', EXPERIMENT_MODEL_ID)
print('RUN_DIR =', RUN_DIR)
print('METRICS_DIR =', METRICS_DIR)
print('CACHE_DIR =', MODEL_CACHE_DIR)
print('REQUEST_DISTRIBUTED_TRAIN =', REQUEST_DISTRIBUTED_TRAIN)
print('VISIBLE_GPU_COUNT =', VISIBLE_GPU_COUNT)
print('DISTRIBUTED_TRAIN =', DISTRIBUTED_TRAIN)
print('NPROC_PER_NODE =', NPROC_PER_NODE)
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', '<default>'))
print('TARGET_MODULES =', TARGET_MODULES)
print('TRAIN_ROWS =', TRAIN_ROWS)
print('SEED =', SEED)
print('VAL_RATIO =', VAL_RATIO)
print('EARLY_STOPPING_PATIENCE =', EARLY_STOPPING_PATIENCE)
print('WEIGHT_DECAY =', WEIGHT_DECAY)
print('WARMUP_RATIO =', WARMUP_RATIO)
print('RUN_EXPERIMENT_QUEUE =', RUN_EXPERIMENT_QUEUE)
print('EXPERIMENT_QUEUE =', EXPERIMENT_QUEUE)
print('SKIP_COMPLETED_IN_QUEUE =', SKIP_COMPLETED_IN_QUEUE)



In [ ]:
# Cell 1: Load token, registry info, and command helpers.
import yaml
import pickle


def _existing_paths(paths):
    out = []
    seen = set()
    for p in paths:
        pp = Path(p).expanduser().resolve()
        if pp in seen:
            continue
        seen.add(pp)
        if pp.exists():
            out.append(pp)
    return out


def resolve_project_root() -> Path:
    candidates = []

    if 'PROJECT_ROOT' in globals():
        candidates.append(Path(PROJECT_ROOT))

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent, cwd.parent.parent])

    candidates.extend([
        Path('/home/rameyjm7/workspace/masked-emotion-lora-benchmark'),
        Path('/home/rameyjm7/workspace/llamafactory-gemma-lora'),
    ])

    for cand in _existing_paths(candidates):
        if (cand / 'configs' / 'model_registry.yaml').exists() and (cand / 'express-emotion-recognition').exists():
            return cand

    raise FileNotFoundError('Could not resolve project root containing configs/model_registry.yaml and express-emotion-recognition')


def load_hf_token(env_path: Path, pkl_path: Path) -> str:
    if pkl_path.exists():
        try:
            with pkl_path.open('rb') as f:
                payload = pickle.load(f)
            token = str(payload.get('HF_TOKEN', '')).strip()
            if token:
                return token
        except Exception as exc:
            print(f'Warning: failed to read {pkl_path}: {exc}')

    if env_path.exists():
        for raw in env_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if line.startswith('HF_TOKEN='):
                token = line.split('=', 1)[1].strip().strip('"').strip("'")
                if token:
                    return token

    raise ValueError(f'HF_TOKEN not found or empty in {pkl_path} or {env_path}')


def run_cmd(cmd, cwd=None, env=None, check=True):
    printable = ' '.join(str(x) for x in cmd)
    print(f'$ {printable}')
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)

    rc = proc.wait()
    out = ''.join(lines)
    if check and rc != 0:
        tail = ''.join(lines[-100:])
        raise RuntimeError(
            f'Command failed ({rc}): {printable}\n'
            f'--- Last output lines ---\n{tail}'
        )
    return out


def parse_eval_metrics(stdout_text):
    metrics = {}
    for key in ['VRate', 'AccL', 'AccV', 'F1V', 'AccV-2']:
        m = re.search(rf'{re.escape(key)}\s*=\s*([0-9]*\.?[0-9]+)', stdout_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics


# Re-resolve paths for the active repo (this notebook may be copied between repos).
PROJECT_ROOT = resolve_project_root()
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
REGISTRY_PATH = PROJECT_ROOT / 'configs' / 'model_registry.yaml'
ENV_PKL_FILE = PROJECT_ROOT / 'configs' / 'env.pkl'
ENV_FILE = PROJECT_ROOT / 'env.txt'

MODEL_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / f'hf_cache_{MODEL_ID}'
RUN_ROOT = PROJECT_ROOT / 'outputs' / 'lora_runs' / f'{MODEL_ID}_experiments'
RUN_DIR = RUN_ROOT / EXPERIMENT_NAME
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics' / f'{MODEL_ID}_experiments'
METRICS_DIR = METRICS_ROOT / EXPERIMENT_NAME
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / f'{MODEL_ID}_experiments'

EVAL_INPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_baseline_eval_input.csv'
RAW_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_lora_raw.csv'
CLEAN_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_lora_clean.csv'
TRAIN_PREP_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_train_prep_stats.json'
METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'
EXP_SUMMARY_PATH = METRICS_ROOT / 'experiment_summary.csv'
FIGURE_PATH = FIGURES_DIR / f'figure2_with_lora_overlay_{EXPERIMENT_MODEL_ID}.png'

for p in [MODEL_CACHE_DIR, RUN_ROOT, RUN_DIR, METRICS_ROOT, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HF_TOKEN = load_hf_token(ENV_FILE, ENV_PKL_FILE)
os.environ['HF_TOKEN'] = HF_TOKEN

registry = yaml.safe_load(REGISTRY_PATH.read_text(encoding='utf-8'))['models']
registry_map = {m['id']: m for m in registry}
assert MODEL_ID in registry_map, f'{MODEL_ID} not found in {REGISTRY_PATH}'
MODEL_SPEC = registry_map[MODEL_ID]

MODEL_NAME_OR_PATH = MODEL_SPEC['model_name_or_path']
MODEL_SIZE_B = float(MODEL_SPEC['size_b'])

print('Resolved PROJECT_ROOT =', PROJECT_ROOT)
print('Resolved EXPRESS_ROOT =', EXPRESS_ROOT)
print('Loaded HF token using', ENV_PKL_FILE, 'with fallback to', ENV_FILE)
print('Model spec:', MODEL_SPEC)




In [ ]:
# Cell 2: Prepare train/eval datasets with isolated output paths.
import pandas as pd

train_source = EXPRESS_ROOT / 'data' / 'express.csv'
eval_source = EXPRESS_ROOT / 'results' / 'roberta-base.csv'

train_df = pd.read_csv(train_source)
if TRAIN_ROWS is not None:
    train_df = train_df.head(TRAIN_ROWS)
train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

eval_df = pd.read_csv(eval_source)
eval_df = eval_df[['segment', 'labels', 'number_of_labels']].copy()
eval_df.to_csv(EVAL_INPUT_PATH, index=False)

print('Train rows:', len(train_df), 'from', train_source)
print('Eval rows:', len(eval_df), 'from', eval_source)
print('Saved eval input to', EVAL_INPUT_PATH)

display(train_df.head(2))
display(eval_df.head(2))


In [ ]:
# Cell 3: Train RoBERTa LoRA (single GPU or DDP via torchrun).
if RUN_TRAIN:
    import torch

    train_script_path = RUN_DIR / 'train_roberta_lora_worker.py'
    train_csv_path = EXPRESS_ROOT / 'data' / 'express.csv'

    worker_script = textwrap.dedent("""
import os
import ast
import json
import argparse
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
from peft import LoraConfig, get_peft_model


def safe_parse_label_list(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    txt = raw.strip()
    if not txt:
        return None
    try:
        val = ast.literal_eval(txt)
        if isinstance(val, list):
            return [str(x).strip().lower() for x in val]
    except Exception:
        return None
    return None


def single_token_id(tokenizer, lab):
    ids = tokenizer(' ' + str(lab), add_special_tokens=False)['input_ids']
    if len(ids) == 1:
        return int(ids[0])
    ids = tokenizer(str(lab), add_special_tokens=False)['input_ids']
    if len(ids) == 1:
        return int(ids[0])
    return None


def load_lexicon_vectors(breakdowns_path):
    lex_df = pd.read_csv(breakdowns_path)
    vec_cols = [c for c in lex_df.columns if c != 'word']
    word_to_vec = {}
    for row in lex_df.itertuples(index=False):
        word = str(getattr(row, 'word')).strip().lower()
        vec = tuple(int(getattr(row, c)) for c in vec_cols)
        word_to_vec[word] = vec
    return word_to_vec


def build_canonical_label_to_id(tokenizer, word_to_vec):
    # Choose a deterministic single-token representative per emotion vector.
    vec_best = {}
    for word, vec in word_to_vec.items():
        tid = single_token_id(tokenizer, word)
        if tid is None:
            continue
        key = (len(word), word)
        prev = vec_best.get(vec)
        if prev is None or key < prev[0]:
            vec_best[vec] = (key, word, tid)

    label_to_tid = {}
    for word, vec in word_to_vec.items():
        best = vec_best.get(vec)
        if best is not None:
            label_to_tid[word] = int(best[2])
    return label_to_tid


def label_to_token_id(lab, tokenizer, canonical_label_to_tid):
    direct_tid = single_token_id(tokenizer, lab)
    if direct_tid is not None:
        return direct_tid, False
    canonical_tid = canonical_label_to_tid.get(str(lab).strip().lower())
    if canonical_tid is not None:
        return int(canonical_tid), True
    return None, False


def build_examples(df, tokenizer, max_length, canonical_label_to_tid):
    examples = []
    skipped_parse = 0
    skipped_mask_mismatch = 0
    skipped_multi_token = 0
    recovered_with_canonical = 0

    for row in df.itertuples(index=False):
        segment = str(row.segment)
        labels = safe_parse_label_list(row.labels)
        if labels is None:
            skipped_parse += 1
            continue

        enc = tokenizer(segment, truncation=True, max_length=max_length)
        input_ids = enc['input_ids']
        attention_mask = enc['attention_mask']
        mask_positions = [i for i, tok in enumerate(input_ids) if tok == tokenizer.mask_token_id]

        if len(mask_positions) != len(labels):
            skipped_mask_mismatch += 1
            continue

        target_ids = []
        valid = True
        row_used_canonical = False
        for lab in labels:
            tok_id, used_canonical = label_to_token_id(lab, tokenizer, canonical_label_to_tid)
            if tok_id is None:
                valid = False
                break
            target_ids.append(int(tok_id))
            row_used_canonical = row_used_canonical or bool(used_canonical)

        if not valid:
            skipped_multi_token += 1
            continue

        if row_used_canonical:
            recovered_with_canonical += 1

        label_ids = [-100] * len(input_ids)
        for pos, tok in zip(mask_positions, target_ids):
            label_ids[pos] = tok

        examples.append(
            {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': label_ids,
            }
        )

    stats = {
        'total_rows': int(len(df)),
        'kept_rows': int(len(examples)),
        'skipped_parse': int(skipped_parse),
        'skipped_mask_mismatch': int(skipped_mask_mismatch),
        'skipped_multi_token_labels': int(skipped_multi_token),
        'recovered_with_canonical_labels': int(recovered_with_canonical),
        'max_length': int(max_length),
    }
    return examples, stats


class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def collate_builder(tokenizer):
    def collate_fn(features):
        batch = tokenizer.pad(
            {
                'input_ids': [f['input_ids'] for f in features],
                'attention_mask': [f['attention_mask'] for f in features],
            },
            padding=True,
            return_tensors='pt',
        )
        max_len = batch['input_ids'].shape[1]
        labels = []
        for f in features:
            padded = f['labels'] + ([-100] * (max_len - len(f['labels'])))
            labels.append(padded)
        batch['labels'] = torch.tensor(labels, dtype=torch.long)
        return batch

    return collate_fn


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_name_or_path', required=True)
    parser.add_argument('--train_source', required=True)
    parser.add_argument('--breakdowns_path', required=True)
    parser.add_argument('--run_dir', required=True)
    parser.add_argument('--cache_dir', required=True)
    parser.add_argument('--train_prep_path', required=True)
    parser.add_argument('--train_rows', type=int, default=-1)
    parser.add_argument('--max_length', type=int, default=256)
    parser.add_argument('--learning_rate', type=float, default=5e-5)
    parser.add_argument('--epochs', type=float, default=2.0)
    parser.add_argument('--batch_size', type=int, default=16)
    parser.add_argument('--grad_accum', type=int, default=1)
    parser.add_argument('--model_id', required=True)
    parser.add_argument('--lora_r', type=int, default=16)
    parser.add_argument('--lora_alpha', type=int, default=32)
    parser.add_argument('--lora_dropout', type=float, default=0.05)
    parser.add_argument('--target_modules', type=str, default='query,key,value')
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--val_ratio', type=float, default=0.10)
    parser.add_argument('--early_stopping_patience', type=int, default=2)
    parser.add_argument('--early_stopping_threshold', type=float, default=0.0)
    parser.add_argument('--weight_decay', type=float, default=0.01)
    parser.add_argument('--warmup_ratio', type=float, default=0.06)
    parser.add_argument('--save_total_limit', type=int, default=2)
    parser.add_argument('--shuffle_train_rows', action='store_true')
    args = parser.parse_args()

    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        raise RuntimeError('HF_TOKEN is required in environment for training.')

    set_seed(int(args.seed))

    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(
        args.model_name_or_path,
        token=hf_token,
        cache_dir=args.cache_dir,
        use_fast=True,
    )
    if tokenizer.mask_token_id is None:
        raise ValueError('Tokenizer has no mask token; cannot run MLM training.')

    word_to_vec = load_lexicon_vectors(args.breakdowns_path)
    canonical_label_to_tid = build_canonical_label_to_id(tokenizer, word_to_vec)
    print('Canonical label map size:', len(canonical_label_to_tid))

    train_df = pd.read_csv(args.train_source)
    if args.train_rows is not None and args.train_rows >= 0:
        train_df = train_df.head(args.train_rows)
    train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

    if args.shuffle_train_rows:
        train_df = train_df.sample(frac=1.0, random_state=int(args.seed)).reset_index(drop=True)

    val_ratio = float(max(0.0, min(0.5, args.val_ratio)))
    if val_ratio > 0.0 and len(train_df) > 50:
        eval_rows = int(round(len(train_df) * val_ratio))
        eval_rows = max(1, min(eval_rows, len(train_df) - 1))
        eval_df = train_df.iloc[:eval_rows].copy()
        train_df = train_df.iloc[eval_rows:].copy()
    else:
        eval_df = train_df.iloc[0:0].copy()

    train_examples, train_stats = build_examples(train_df, tokenizer, args.max_length, canonical_label_to_tid)
    eval_examples, eval_stats = build_examples(eval_df, tokenizer, args.max_length, canonical_label_to_tid) if len(eval_df) else ([], {
        'total_rows': 0,
        'kept_rows': 0,
        'skipped_parse': 0,
        'skipped_mask_mismatch': 0,
        'skipped_multi_token_labels': 0,
        'max_length': int(args.max_length),
    })

    rank = int(os.environ.get('RANK', '0'))
    world_size = int(os.environ.get('WORLD_SIZE', '1'))
    prep_stats = {
        'seed': int(args.seed),
        'val_ratio': float(val_ratio),
        'canonical_label_map_size': int(len(canonical_label_to_tid)),
        'train': train_stats,
        'eval': eval_stats,
    }
    if rank == 0:
        Path(args.train_prep_path).write_text(json.dumps(prep_stats, indent=2), encoding='utf-8')
        print('Training prep stats:', prep_stats)

    if not train_examples:
        raise RuntimeError('No valid training examples were built.')

    train_dataset = ListDataset(train_examples)
    eval_dataset = ListDataset(eval_examples) if len(eval_examples) > 0 else None
    has_eval = eval_dataset is not None

    collate_fn = collate_builder(tokenizer)

    base_model = AutoModelForMaskedLM.from_pretrained(
        args.model_name_or_path,
        token=hf_token,
        cache_dir=args.cache_dir,
    )

    parsed_target_modules = [m.strip() for m in str(args.target_modules).split(',') if m.strip()]
    if not parsed_target_modules:
        parsed_target_modules = ['query', 'key', 'value']

    lora_cfg = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias='none',
        target_modules=parsed_target_modules,
    )
    model = get_peft_model(base_model, lora_cfg)
    if rank == 0:
        model.print_trainable_parameters()

    use_cuda = torch.cuda.is_available()
    use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())
    use_fp16 = bool(use_cuda and not use_bf16)

    trainer_args = TrainingArguments(
        output_dir=str(run_dir / 'trainer_state'),
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=max(1, args.batch_size),
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.learning_rate,
        num_train_epochs=args.epochs,
        logging_steps=20,
        save_strategy='epoch',
        eval_strategy='epoch' if has_eval else 'no',
        load_best_model_at_end=bool(has_eval),
        metric_for_best_model='eval_loss' if has_eval else None,
        greater_is_better=False if has_eval else None,
        save_total_limit=max(1, int(args.save_total_limit)),
        weight_decay=float(args.weight_decay),
        warmup_ratio=float(args.warmup_ratio),
        remove_unused_columns=False,
        fp16=use_fp16,
        bf16=use_bf16,
        ddp_find_unused_parameters=False,
        report_to=[],
        seed=int(args.seed),
        data_seed=int(args.seed),
    )

    callbacks = []
    if has_eval and int(args.early_stopping_patience) > 0:
        callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=int(args.early_stopping_patience),
                early_stopping_threshold=float(args.early_stopping_threshold),
            )
        )

    trainer = Trainer(
        model=model,
        args=trainer_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
        callbacks=callbacks,
    )

    train_out = trainer.train()

    best_checkpoint = trainer.state.best_model_checkpoint if has_eval else None

    if trainer.is_world_process_zero():
        trainer.model.save_pretrained(str(run_dir))
        tokenizer.save_pretrained(str(run_dir))

        summary = {
            'train_runtime': float(getattr(train_out, 'metrics', {}).get('train_runtime', 0.0)),
            'train_samples': int(len(train_dataset)),
            'eval_samples': int(len(eval_dataset)) if has_eval else 0,
            'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
            'model_id': args.model_id,
            'lora_r': args.lora_r,
            'lora_alpha': args.lora_alpha,
            'lora_dropout': args.lora_dropout,
            'target_modules': [m.strip() for m in str(args.target_modules).split(',') if m.strip()],
            'model_name_or_path': args.model_name_or_path,
            'world_size': world_size,
            'seed': int(args.seed),
            'val_ratio': float(val_ratio),
            'early_stopping_patience': int(args.early_stopping_patience),
            'early_stopping_threshold': float(args.early_stopping_threshold),
            'best_model_checkpoint': best_checkpoint,
        }
        (run_dir / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
        print('Best checkpoint:', best_checkpoint)
        print('Saved LoRA adapter to:', run_dir)


if __name__ == '__main__':
    main()
    """).strip() + '\n'

    train_script_path.write_text(worker_script, encoding='utf-8')
    print('Wrote training worker script to:', train_script_path)

    train_rows_arg = -1 if TRAIN_ROWS is None else int(TRAIN_ROWS)

    common_args = [
        '--model_name_or_path', str(MODEL_NAME_OR_PATH),
        '--train_source', str(train_csv_path),
        '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
        '--run_dir', str(RUN_DIR),
        '--cache_dir', str(MODEL_CACHE_DIR),
        '--train_prep_path', str(TRAIN_PREP_PATH),
        '--train_rows', str(train_rows_arg),
        '--max_length', str(MAX_LENGTH),
        '--learning_rate', str(LR),
        '--epochs', str(EPOCHS),
        '--batch_size', str(TRAIN_BATCH_SIZE),
        '--grad_accum', str(GRAD_ACCUM),
        '--model_id', str(EXPERIMENT_MODEL_ID),
        '--lora_r', str(LORA_R),
        '--lora_alpha', str(LORA_ALPHA),
        '--lora_dropout', str(LORA_DROPOUT),
        '--target_modules', ','.join(TARGET_MODULES),
        '--seed', str(SEED),
        '--val_ratio', str(VAL_RATIO),
        '--early_stopping_patience', str(EARLY_STOPPING_PATIENCE),
        '--early_stopping_threshold', str(EARLY_STOPPING_THRESHOLD),
        '--weight_decay', str(WEIGHT_DECAY),
        '--warmup_ratio', str(WARMUP_RATIO),
        '--save_total_limit', str(SAVE_TOTAL_LIMIT),
    ]
    if SHUFFLE_TRAIN_ROWS:
        common_args.append('--shuffle_train_rows')

    env = {
        'HF_TOKEN': HF_TOKEN,
        'TOKENIZERS_PARALLELISM': 'false',
    }

    if DISTRIBUTED_TRAIN:
        if NPROC_PER_NODE < 2:
            raise ValueError('DISTRIBUTED_TRAIN=True requires NPROC_PER_NODE >= 2')
        cmd = [
            'torchrun',
            '--standalone',
            f'--nproc_per_node={NPROC_PER_NODE}',
            f'--master_port={MASTER_PORT}',
            str(train_script_path),
            *common_args,
        ]
    else:
        cmd = ['python', str(train_script_path), *common_args]

    try:
        _ = run_cmd(cmd, cwd=PROJECT_ROOT, env=env, check=True)
    except RuntimeError as exc:
        msg = str(exc)
        duplicate_gpu = ('Duplicate GPU detected' in msg) or ('ncclInvalidUsage' in msg)
        if DISTRIBUTED_TRAIN and duplicate_gpu:
            print('DDP failed with duplicate GPU mapping; falling back to single-process training.')
            fallback_cmd = ['python', str(train_script_path), *common_args]
            _ = run_cmd(fallback_cmd, cwd=PROJECT_ROOT, env=env, check=True)
        else:
            raise
else:
    print('Training skipped (RUN_TRAIN=False).')






In [ ]:
# Cell 4: Run RoBERTa LoRA inference on baseline-comparable rows.
if RUN_INFERENCE:
    import torch
    from tqdm.auto import tqdm
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    from peft import PeftModel

    def resolve_adapter_path(run_dir: Path) -> Path:
        summary_path = run_dir / 'training_summary.json'
        if summary_path.exists():
            try:
                obj = json.loads(summary_path.read_text(encoding='utf-8'))
                ckpt = obj.get('best_model_checkpoint')
                if ckpt:
                    ckpt_path = Path(str(ckpt))
                    if ckpt_path.exists():
                        return ckpt_path
            except Exception as exc:
                print(f'Warning: failed to parse {summary_path}: {exc}')

        state_path = run_dir / 'trainer_state' / 'trainer_state.json'
        if state_path.exists():
            try:
                obj = json.loads(state_path.read_text(encoding='utf-8'))
                ckpt = obj.get('best_model_checkpoint')
                if ckpt:
                    ckpt_path = Path(str(ckpt))
                    if ckpt_path.exists():
                        return ckpt_path
            except Exception as exc:
                print(f'Warning: failed to parse {state_path}: {exc}')

        return run_dir

    eval_df = pd.read_csv(EVAL_INPUT_PATH)
    adapter_path = resolve_adapter_path(RUN_DIR)
    print('Using adapter path:', adapter_path)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
    )
    base_model = AutoModelForMaskedLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    model = PeftModel.from_pretrained(base_model, str(adapter_path))

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()

    mask_token = tokenizer.mask_token
    if mask_token is None:
        raise ValueError('Tokenizer has no mask token; cannot run MLM inference.')

    def normalize_pred_token(text):
        txt = re.sub(r'\s+', ' ', str(text)).strip().lower()
        return txt

    def predict_mask_list(segment, num_labels):
        target_n = int(num_labels)

        enc = tokenizer(
            str(segment),
            return_tensors='pt',
            truncation=True,
            max_length=MAX_LENGTH,
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        mask_positions = (enc['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=False).flatten().tolist()
        if len(mask_positions) != target_n:
            return 'INVALID OUTPUT'

        with torch.inference_mode():
            logits = model(**enc).logits[0]

        preds = []
        for pos in mask_positions:
            pred_id = int(torch.argmax(logits[int(pos)]).item())
            pred_word = normalize_pred_token(tokenizer.decode([pred_id], skip_special_tokens=True))
            if not pred_word:
                pred_word = 'neutral'
            preds.append(pred_word)

        if len(preds) != target_n:
            return 'INVALID OUTPUT'
        return str(preds)

    outputs = []
    for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc='RoBERTa LoRA inference'):
        outputs.append(predict_mask_list(row.segment, row.number_of_labels))

    raw_df = eval_df.copy()
    raw_df['output'] = outputs
    raw_df.to_csv(RAW_OUTPUT_PATH, index=False)

    print('Saved raw outputs to:', RAW_OUTPUT_PATH)
    print('Rows:', len(raw_df))
else:
    print('Inference skipped (RUN_INFERENCE=False).')



In [ ]:
# Cell 5: Clean and evaluate outputs with existing project scripts.
lora_metrics = {}
if RUN_EVAL:
    _ = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
            '--input', str(RAW_OUTPUT_PATH),
            '--output', str(CLEAN_OUTPUT_PATH),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    eval_stdout = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
            '--result_path', str(CLEAN_OUTPUT_PATH),
            '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    lora_metrics = parse_eval_metrics(eval_stdout)
    print('LoRA metrics:', lora_metrics)
else:
    print('Evaluation skipped (RUN_EVAL=False).')


In [ ]:
# Cell 6: Upsert shared metrics table for cross-model comparison.
import pandas as pd

row = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'model_id': EXPERIMENT_MODEL_ID,
    'family': MODEL_SPEC['family'],
    'model_name_or_path': MODEL_NAME_OR_PATH,
    'size_b': MODEL_SIZE_B,
    'pipeline': MODEL_SPEC['pipeline'],
    'is_lora': True,
    'v_rate': lora_metrics.get('VRate'),
    'accl': lora_metrics.get('AccL'),
    'accv': lora_metrics.get('AccV'),
    'f1v': lora_metrics.get('F1V'),
    'accv2': lora_metrics.get('AccV-2'),
    'clean_output_path': str(CLEAN_OUTPUT_PATH),
    'run_dir': str(RUN_DIR),
}

if METRICS_TABLE_PATH.exists():
    metrics_df = pd.read_csv(METRICS_TABLE_PATH)
else:
    metrics_df = pd.DataFrame()

if not metrics_df.empty and {'model_id', 'is_lora'}.issubset(metrics_df.columns):
    metrics_df = metrics_df[~((metrics_df['model_id'] == EXPERIMENT_MODEL_ID) & (metrics_df['is_lora'].astype(str).str.lower() == 'true'))]

metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)

print('Updated metrics table:', METRICS_TABLE_PATH)
display(metrics_df.tail(10))


summary_row = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'base_model_id': MODEL_ID,
    'experiment_model_id': EXPERIMENT_MODEL_ID,
    'experiment_name': EXPERIMENT_NAME,
    'train_rows': TRAIN_ROWS,
    'max_length': MAX_LENGTH,
    'lr': LR,
    'epochs': EPOCHS,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'lora_dropout': LORA_DROPOUT,
    'target_modules': ','.join(TARGET_MODULES),
    'v_rate': lora_metrics.get('VRate'),
    'accl': lora_metrics.get('AccL'),
    'accv': lora_metrics.get('AccV'),
    'f1v': lora_metrics.get('F1V'),
    'accv2': lora_metrics.get('AccV-2'),
    'clean_output_path': str(CLEAN_OUTPUT_PATH),
    'run_dir': str(RUN_DIR),
}

if EXP_SUMMARY_PATH.exists():
    exp_df = pd.read_csv(EXP_SUMMARY_PATH)
else:
    exp_df = pd.DataFrame()

if not exp_df.empty and {'experiment_model_id'}.issubset(exp_df.columns):
    exp_df = exp_df[exp_df['experiment_model_id'] != EXPERIMENT_MODEL_ID]

exp_df = pd.concat([exp_df, pd.DataFrame([summary_row])], ignore_index=True)
exp_df.to_csv(EXP_SUMMARY_PATH, index=False)
print('Updated experiment summary:', EXP_SUMMARY_PATH)
display(exp_df.tail(20))


In [ ]:
# Cell 7: Plot Figure-2 baseline points with RoBERTa LoRA overlay.
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

lex_df = pd.read_csv(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv')
vec_map = {str(r['word']).lower(): tuple(int(v) for v in r.iloc[1:].tolist()) for _, r in lex_df.iterrows()}
zero_vec = tuple([0] * 10)

_parse_cache = {}
def parse_list_cached(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if s == 'INVALID OUTPUT':
        return None
    if s in _parse_cache:
        return _parse_cache[s]
    try:
        v = ast.literal_eval(s)
        parsed = [str(x).strip().lower() for x in v] if isinstance(v, list) else None
    except Exception:
        parsed = None
    _parse_cache[s] = parsed
    return parsed


def compute_accv(csv_path):
    df = pd.read_csv(csv_path, usecols=['labels', 'output_formatted', 'number_of_labels'])
    total_masks = int(df['number_of_labels'].sum())
    matches = 0
    for raw_labels, raw_preds in zip(df['labels'].tolist(), df['output_formatted'].tolist()):
        labels = parse_list_cached(raw_labels)
        preds = parse_list_cached(raw_preds)
        if labels is None or preds is None:
            continue
        for t, p in zip(labels, preds):
            matches += int(vec_map.get(t, zero_vec) == vec_map.get(p, zero_vec))
    return (matches / total_masks) if total_masks > 0 else float('nan')

baseline_models = [
    {'label': 'Flan-T5-large', 'family': 'Flan-T5', 'size_b': 0.78, 'file': 'flan_t5_large.csv'},
    {'label': 'Flan-T5-xl', 'family': 'Flan-T5', 'size_b': 3.0, 'file': 'flan_t5_xl.csv'},
    {'label': 'Flan-T5-xxl', 'family': 'Flan-T5', 'size_b': 11.0, 'file': 'flan_t5_xxl.csv'},
    {'label': 'Gpt-3.5-turbo', 'family': 'GPT', 'size_b': 175.0, 'file': 'gpt_35_turbo_0125.csv'},
    {'label': 'Gpt-4o', 'family': 'GPT', 'size_b': 1750.0, 'file': 'gpt_4o.csv'},
    {'label': 'Gemma-2-2B-it', 'family': 'Gemma2', 'size_b': 2.0, 'file': 'gemma2_2B_it.csv'},
    {'label': 'Gemma-2-9B-it', 'family': 'Gemma2', 'size_b': 9.0, 'file': 'gemma2_9B_it.csv'},
    {'label': 'Gemma-2-27B-it', 'family': 'Gemma2', 'size_b': 27.0, 'file': 'gemma2_27B_it.csv'},
    {'label': 'Llama3.1-8B-instruct', 'family': 'Llama3.1', 'size_b': 8.0, 'file': 'llama3.1_8B_instruct.csv'},
    {'label': 'Llama3.1-70B-instruct', 'family': 'Llama3.1', 'size_b': 70.0, 'file': 'llama3.1_70B_instruct.csv'},
    {'label': 'Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'longformer.csv'},
    {'label': 'Mental-Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'mental-longformer.csv'},
    {'label': 'Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'roberta-base.csv'},
    {'label': 'Mental-Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'mental-roberta-base.csv'},
]

rows = []
for m in baseline_models:
    p = EXPRESS_ROOT / 'results' / m['file']
    rows.append({**m, 'accv': compute_accv(p)})
fig_df = pd.DataFrame(rows)

if CLEAN_OUTPUT_PATH.exists():
    lora_accv = compute_accv(CLEAN_OUTPUT_PATH)
else:
    lora_accv = None

sns.set_theme(style='whitegrid')
family_colors = {
    'Flan-T5': '#4C72B0',
    'GPT': '#DD8452',
    'Gemma2': '#55A868',
    'Llama3.1': '#C44E52',
    'Longformer': '#8172B2',
    'Roberta': '#937860',
}
family_markers = {
    'Flan-T5': 'D',
    'GPT': 'P',
    'Gemma2': '^',
    'Llama3.1': 'X',
    'Longformer': 's',
    'Roberta': 'o',
}

plt.figure(figsize=(14, 6))
for fam in ['Flan-T5', 'GPT', 'Gemma2', 'Llama3.1', 'Longformer', 'Roberta']:
    sub = fig_df[fig_df['family'] == fam].sort_values('size_b')
    if sub.empty:
        continue
    plt.plot(
        sub['size_b'], sub['accv'],
        marker=family_markers[fam],
        color=family_colors[fam],
        linestyle='--',
        linewidth=1.25,
        markersize=8,
        label=fam,
    )

if lora_accv is not None:
    plt.scatter([MODEL_SIZE_B], [lora_accv], marker='*', s=320, color='black', label=f'Roberta-base + LoRA ({EXPERIMENT_NAME})')
    plt.text(MODEL_SIZE_B * 1.15, lora_accv + 0.004, f'Roberta-base + LoRA ({EXPERIMENT_NAME})', fontsize=10)
    print(f'RoBERTa LoRA AccV ({EXPERIMENT_NAME}):', lora_accv)
else:
    print('No LoRA clean output found; run cells 4-5 first.')

plt.xscale('log')
plt.xlabel('Model Size')
plt.ylabel('Accuracy (Vector)')
y_max = 0.40
if lora_accv is not None:
    y_max = max(y_max, float(lora_accv) + 0.03)
plt.ylim(0.08, 0.5)
plt.xticks([0.1, 1, 10, 100, 1000], ['100M', '1B', '10B', '100B', '1000B'])
plt.title(f'Figure 2 Baseline + RoBERTa LoRA Result ({EXPERIMENT_NAME})')
plt.legend(loc='lower left')
plt.tight_layout()

plt.savefig(FIGURE_PATH, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', FIGURE_PATH)



In [ ]:
# Cell 8: Auto-run queued experiments (e.g., 1 then 2) and persist metrics.
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM
from peft import PeftModel


def resolve_adapter_path_for_run(run_dir: Path) -> Path:
    summary_path = run_dir / 'training_summary.json'
    if summary_path.exists():
        try:
            obj = json.loads(summary_path.read_text(encoding='utf-8'))
            ckpt = obj.get('best_model_checkpoint')
            if ckpt:
                ckpt_path = Path(str(ckpt))
                if ckpt_path.exists():
                    return ckpt_path
        except Exception as exc:
            print(f'Warning: failed to parse {summary_path}: {exc}')

    state_path = run_dir / 'trainer_state' / 'trainer_state.json'
    if state_path.exists():
        try:
            obj = json.loads(state_path.read_text(encoding='utf-8'))
            ckpt = obj.get('best_model_checkpoint')
            if ckpt:
                ckpt_path = Path(str(ckpt))
                if ckpt_path.exists():
                    return ckpt_path
        except Exception as exc:
            print(f'Warning: failed to parse {state_path}: {exc}')

    return run_dir


if RUN_EXPERIMENT_QUEUE:
    def script_supports_extended_args(path: Path) -> bool:
        try:
            txt = path.read_text(encoding='utf-8')
        except Exception:
            return False
        required = [
            '--seed',
            '--val_ratio',
            '--early_stopping_patience',
            '--early_stopping_threshold',
            '--weight_decay',
            '--warmup_ratio',
            '--save_total_limit',
            '--shuffle_train_rows',
        ]
        return all(flag in txt for flag in required)

    preferred_worker = RUN_DIR / 'train_roberta_lora_worker.py'
    discovered = sorted(RUN_ROOT.glob('*/train_roberta_lora_worker.py'), key=lambda x: x.stat().st_mtime, reverse=True)

    candidates = []
    if preferred_worker.exists():
        candidates.append(preferred_worker)
    candidates.extend(discovered)

    deduped = []
    seen = set()
    for c in candidates:
        cc = Path(c)
        if cc in seen:
            continue
        seen.add(cc)
        deduped.append(cc)

    if not deduped:
        raise FileNotFoundError(
            f'No worker script found under {RUN_ROOT}. Run Cell 3 first to generate it.'
        )

    worker_script_path = None
    for cand in deduped:
        if script_supports_extended_args(cand):
            worker_script_path = cand
            break

    if worker_script_path is None:
        raise RuntimeError(
            'Found worker scripts, but none support the extended args used by Cell 8. '
            'Run Cell 3 once more to regenerate a fresh worker script, then rerun Cell 8.'
        )

    print('Using worker script:', worker_script_path)

    eval_source = EXPRESS_ROOT / 'results' / 'roberta-base.csv'
    eval_df_base = pd.read_csv(eval_source)[['segment', 'labels', 'number_of_labels']].copy()

    summary_rows = []

    for exp_idx in EXPERIMENT_QUEUE:
        if not (0 <= int(exp_idx) < len(EXPERIMENTS)):
            print(f'Skipping invalid experiment index: {exp_idx}')
            continue

        exp = EXPERIMENTS[int(exp_idx)]
        exp_name = exp['name']
        exp_model_id = f"{MODEL_ID}_{exp_name}"

        exp_run_dir = RUN_ROOT / exp_name
        exp_metrics_dir = METRICS_ROOT / exp_name
        exp_run_dir.mkdir(parents=True, exist_ok=True)
        exp_metrics_dir.mkdir(parents=True, exist_ok=True)

        exp_eval_input = exp_metrics_dir / f'{exp_model_id}_baseline_eval_input.csv'
        exp_raw_output = exp_metrics_dir / f'{exp_model_id}_lora_raw.csv'
        exp_clean_output = exp_metrics_dir / f'{exp_model_id}_lora_clean.csv'
        exp_train_prep = exp_metrics_dir / f'{exp_model_id}_train_prep_stats.json'

        if SKIP_COMPLETED_IN_QUEUE and exp_clean_output.exists():
            print(f'Skipping {exp_name}: clean output already exists at {exp_clean_output}')
            continue

        eval_df_base.to_csv(exp_eval_input, index=False)

        common_args = [
            '--model_name_or_path', str(MODEL_NAME_OR_PATH),
            '--train_source', str(EXPRESS_ROOT / 'data' / 'express.csv'),
            '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
            '--run_dir', str(exp_run_dir),
            '--cache_dir', str(MODEL_CACHE_DIR),
            '--train_prep_path', str(exp_train_prep),
            '--train_rows', str(-1 if exp['train_rows'] is None else int(exp['train_rows'])),
            '--max_length', str(int(exp['max_length'])),
            '--learning_rate', str(float(exp['lr'])),
            '--epochs', str(float(exp['epochs'])),
            '--batch_size', str(int(exp['train_batch_size'])),
            '--grad_accum', str(int(exp['grad_accum'])),
            '--model_id', exp_model_id,
            '--lora_r', str(int(exp['lora_r'])),
            '--lora_alpha', str(int(exp['lora_alpha'])),
            '--lora_dropout', str(float(exp['lora_dropout'])),
            '--target_modules', ','.join(exp['target_modules']),
            '--seed', str(int(exp.get('seed', 42))),
            '--val_ratio', str(float(exp.get('val_ratio', 0.10))),
            '--early_stopping_patience', str(int(exp.get('early_stopping_patience', 2))),
            '--early_stopping_threshold', str(float(exp.get('early_stopping_threshold', 0.0))),
            '--weight_decay', str(float(exp.get('weight_decay', 0.01))),
            '--warmup_ratio', str(float(exp.get('warmup_ratio', 0.06))),
            '--save_total_limit', str(int(exp.get('save_total_limit', 2))),
        ]
        if bool(exp.get('shuffle_train_rows', True)):
            common_args.append('--shuffle_train_rows')

        env = {'HF_TOKEN': HF_TOKEN, 'TOKENIZERS_PARALLELISM': 'false'}

        if DISTRIBUTED_TRAIN and NPROC_PER_NODE >= 2:
            train_cmd = [
                'torchrun',
                '--standalone',
                f'--nproc_per_node={NPROC_PER_NODE}',
                f'--master_port={MASTER_PORT + int(exp_idx)}',
                str(worker_script_path),
                *common_args,
            ]
        else:
            train_cmd = ['python', str(worker_script_path), *common_args]

        print(f'=== Training {exp_name} ===')
        try:
            _ = run_cmd(train_cmd, cwd=PROJECT_ROOT, env=env, check=True)
        except RuntimeError as exc:
            msg = str(exc)
            duplicate_gpu = ('Duplicate GPU detected' in msg) or ('ncclInvalidUsage' in msg)
            if DISTRIBUTED_TRAIN and duplicate_gpu:
                print('DDP duplicate-GPU issue detected; retrying single-process training.')
                _ = run_cmd(['python', str(worker_script_path), *common_args], cwd=PROJECT_ROOT, env=env, check=True)
            else:
                raise

        print(f'=== Inference {exp_name} ===')
        if exp_raw_output.exists():
            exp_raw_output.unlink()

        adapter_path = resolve_adapter_path_for_run(exp_run_dir)
        print('Using adapter path for inference:', adapter_path)

        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME_OR_PATH,
            token=HF_TOKEN,
            cache_dir=str(MODEL_CACHE_DIR),
            use_fast=True,
        )
        base_model = AutoModelForMaskedLM.from_pretrained(
            MODEL_NAME_OR_PATH,
            token=HF_TOKEN,
            cache_dir=str(MODEL_CACHE_DIR),
        )
        model = PeftModel.from_pretrained(base_model, str(adapter_path))

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = model.to(device)
        model.eval()

        mask_token = tokenizer.mask_token
        if mask_token is None:
            raise ValueError('Tokenizer has no mask token; cannot run MLM inference.')

        def normalize_pred_token(text):
            txt = re.sub(r'\s+', ' ', str(text)).strip().lower()
            return txt

        def predict_mask_list(segment, num_labels):
            target_n = int(num_labels)

            enc = tokenizer(
                str(segment),
                return_tensors='pt',
                truncation=True,
                max_length=int(exp['max_length']),
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            mask_positions = (enc['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=False).flatten().tolist()
            if len(mask_positions) != target_n:
                return 'INVALID OUTPUT'

            with torch.inference_mode():
                logits = model(**enc).logits[0]

            preds = []
            for pos in mask_positions:
                pred_id = int(torch.argmax(logits[int(pos)]).item())
                pred_word = normalize_pred_token(tokenizer.decode([pred_id], skip_special_tokens=True))
                if not pred_word:
                    pred_word = 'neutral'
                preds.append(pred_word)

            if len(preds) != target_n:
                return 'INVALID OUTPUT'
            return str(preds)

        eval_df = pd.read_csv(exp_eval_input)
        outputs = []
        for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc=f'RoBERTa LoRA inference ({exp_name})'):
            outputs.append(predict_mask_list(row.segment, row.number_of_labels))

        raw_df = eval_df.copy()
        raw_df['output'] = outputs
        raw_df.to_csv(exp_raw_output, index=False)

        _ = run_cmd([
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
            '--input', str(exp_raw_output),
            '--output', str(exp_clean_output),
        ], cwd=PROJECT_ROOT, check=True)

        eval_stdout = run_cmd([
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
            '--result_path', str(exp_clean_output),
            '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
        ], cwd=PROJECT_ROOT, check=True)

        exp_metrics = parse_eval_metrics(eval_stdout)
        print(f'LoRA metrics ({exp_name}):', exp_metrics)

        summary_rows.append({
            'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
            'base_model_id': MODEL_ID,
            'experiment_model_id': exp_model_id,
            'experiment_name': exp_name,
            'train_rows': exp['train_rows'],
            'max_length': exp['max_length'],
            'lr': exp['lr'],
            'epochs': exp['epochs'],
            'train_batch_size': exp['train_batch_size'],
            'grad_accum': exp['grad_accum'],
            'lora_r': exp['lora_r'],
            'lora_alpha': exp['lora_alpha'],
            'lora_dropout': exp['lora_dropout'],
            'target_modules': ','.join(exp['target_modules']),
            'seed': int(exp.get('seed', 42)),
            'val_ratio': float(exp.get('val_ratio', 0.10)),
            'early_stopping_patience': int(exp.get('early_stopping_patience', 2)),
            'weight_decay': float(exp.get('weight_decay', 0.01)),
            'warmup_ratio': float(exp.get('warmup_ratio', 0.06)),
            'v_rate': exp_metrics.get('VRate'),
            'accl': exp_metrics.get('AccL'),
            'accv': exp_metrics.get('AccV'),
            'f1v': exp_metrics.get('F1V'),
            'accv2': exp_metrics.get('AccV-2'),
            'clean_output_path': str(exp_clean_output),
            'run_dir': str(exp_run_dir),
            'adapter_path': str(adapter_path),
        })

    if summary_rows:
        new_df = pd.DataFrame(summary_rows)

        if EXP_SUMMARY_PATH.exists():
            exp_df = pd.read_csv(EXP_SUMMARY_PATH)
        else:
            exp_df = pd.DataFrame()

        for exp_model_id in new_df['experiment_model_id'].tolist():
            if not exp_df.empty and 'experiment_model_id' in exp_df.columns:
                exp_df = exp_df[exp_df['experiment_model_id'] != exp_model_id]
        exp_df = pd.concat([exp_df, new_df], ignore_index=True)
        exp_df.to_csv(EXP_SUMMARY_PATH, index=False)
        print('Updated experiment summary:', EXP_SUMMARY_PATH)
        display(exp_df.sort_values('timestamp_utc').tail(20))

        if METRICS_TABLE_PATH.exists():
            all_df = pd.read_csv(METRICS_TABLE_PATH)
        else:
            all_df = pd.DataFrame()

        for _, row in new_df.iterrows():
            metric_row = {
                'timestamp_utc': row['timestamp_utc'],
                'model_id': row['experiment_model_id'],
                'family': MODEL_SPEC['family'],
                'model_name_or_path': MODEL_NAME_OR_PATH,
                'size_b': MODEL_SIZE_B,
                'pipeline': MODEL_SPEC['pipeline'],
                'is_lora': True,
                'v_rate': row['v_rate'],
                'accl': row['accl'],
                'accv': row['accv'],
                'f1v': row['f1v'],
                'accv2': row['accv2'],
                'clean_output_path': row['clean_output_path'],
                'run_dir': row['run_dir'],
            }
            if not all_df.empty and {'model_id', 'is_lora'}.issubset(all_df.columns):
                all_df = all_df[~((all_df['model_id'] == metric_row['model_id']) & (all_df['is_lora'].astype(str).str.lower() == 'true'))]
            all_df = pd.concat([all_df, pd.DataFrame([metric_row])], ignore_index=True)

        all_df.to_csv(METRICS_TABLE_PATH, index=False)
        print('Updated shared metrics table:', METRICS_TABLE_PATH)
    else:
        print('Queue completed with no new runs (all skipped or empty queue).')
else:
    print('Queue run skipped (RUN_EXPERIMENT_QUEUE=False).')




In [ ]:
# Cell 9: Plot Figure 2 with all completed RoBERTa experiment points.
import seaborn as sns
import matplotlib.pyplot as plt

lex_df = pd.read_csv(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv')
vec_map = {str(r['word']).lower(): tuple(int(v) for v in r.iloc[1:].tolist()) for _, r in lex_df.iterrows()}
zero_vec = tuple([0] * 10)

_parse_cache = {}
def parse_list_cached(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if s == 'INVALID OUTPUT':
        return None
    if s in _parse_cache:
        return _parse_cache[s]
    try:
        v = ast.literal_eval(s)
        parsed = [str(x).strip().lower() for x in v] if isinstance(v, list) else None
    except Exception:
        parsed = None
    _parse_cache[s] = parsed
    return parsed


def compute_accv(csv_path):
    df = pd.read_csv(csv_path, usecols=['labels', 'output_formatted', 'number_of_labels'])
    total_masks = int(df['number_of_labels'].sum())
    matches = 0
    for raw_labels, raw_preds in zip(df['labels'].tolist(), df['output_formatted'].tolist()):
        labels = parse_list_cached(raw_labels)
        preds = parse_list_cached(raw_preds)
        if labels is None or preds is None:
            continue
        for t, p in zip(labels, preds):
            matches += int(vec_map.get(t, zero_vec) == vec_map.get(p, zero_vec))
    return (matches / total_masks) if total_masks > 0 else float('nan')

baseline_models = [
    {'label': 'Flan-T5-large', 'family': 'Flan-T5', 'size_b': 0.78, 'file': 'flan_t5_large.csv'},
    {'label': 'Flan-T5-xl', 'family': 'Flan-T5', 'size_b': 3.0, 'file': 'flan_t5_xl.csv'},
    {'label': 'Flan-T5-xxl', 'family': 'Flan-T5', 'size_b': 11.0, 'file': 'flan_t5_xxl.csv'},
    {'label': 'Gpt-3.5-turbo', 'family': 'GPT', 'size_b': 175.0, 'file': 'gpt_35_turbo_0125.csv'},
    {'label': 'Gpt-4o', 'family': 'GPT', 'size_b': 1750.0, 'file': 'gpt_4o.csv'},
    {'label': 'Gemma-2-2B-it', 'family': 'Gemma2', 'size_b': 2.0, 'file': 'gemma2_2B_it.csv'},
    {'label': 'Gemma-2-9B-it', 'family': 'Gemma2', 'size_b': 9.0, 'file': 'gemma2_9B_it.csv'},
    {'label': 'Gemma-2-27B-it', 'family': 'Gemma2', 'size_b': 27.0, 'file': 'gemma2_27B_it.csv'},
    {'label': 'Llama3.1-8B-instruct', 'family': 'Llama3.1', 'size_b': 8.0, 'file': 'llama3.1_8B_instruct.csv'},
    {'label': 'Llama3.1-70B-instruct', 'family': 'Llama3.1', 'size_b': 70.0, 'file': 'llama3.1_70B_instruct.csv'},
    {'label': 'Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'longformer.csv'},
    {'label': 'Mental-Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'mental-longformer.csv'},
    {'label': 'Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'roberta-base.csv'},
    {'label': 'Mental-Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'mental-roberta-base.csv'},
]

rows = []
for m in baseline_models:
    p_csv = EXPRESS_ROOT / 'results' / m['file']
    rows.append({**m, 'accv': compute_accv(p_csv)})
fig_df = pd.DataFrame(rows)

exp_points = []
if EXP_SUMMARY_PATH.exists():
    exp_df = pd.read_csv(EXP_SUMMARY_PATH)
    exp_df = exp_df.sort_values('timestamp_utc').drop_duplicates('experiment_name', keep='last')
    for _, r in exp_df.iterrows():
        if pd.notna(r.get('accv')):
            exp_points.append({'name': r['experiment_name'], 'accv': float(r['accv'])})

sns.set_theme(style='whitegrid')
family_colors = {
    'Flan-T5': '#4C72B0',
    'GPT': '#DD8452',
    'Gemma2': '#55A868',
    'Llama3.1': '#C44E52',
    'Longformer': '#8172B2',
    'Roberta': '#937860',
}
family_markers = {
    'Flan-T5': 'D',
    'GPT': 'P',
    'Gemma2': '^',
    'Llama3.1': 'X',
    'Longformer': 's',
    'Roberta': 'o',
}

# Dynamic y-range so new high-scoring configs are never clipped.
all_y = [float(v) for v in fig_df['accv'].dropna().tolist()]
all_y.extend([float(ep['accv']) for ep in exp_points if pd.notna(ep.get('accv'))])
if all_y:
    y_min = max(0.0, min(all_y) - 0.02)
    y_max = max(0.5, max(all_y) + 0.03)
else:
    y_min, y_max = 0.08, 0.5

plt.figure(figsize=(14, 6))
for fam in ['Flan-T5', 'GPT', 'Gemma2', 'Llama3.1', 'Longformer', 'Roberta']:
    sub = fig_df[fig_df['family'] == fam].sort_values('size_b')
    if sub.empty:
        continue
    plt.plot(
        sub['size_b'], sub['accv'],
        marker=family_markers[fam],
        color=family_colors[fam],
        linestyle='--',
        linewidth=1.25,
        markersize=8,
        label=fam,
    )

star_colors = ['black', '#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']
for i, ep in enumerate(exp_points):
    c = star_colors[i % len(star_colors)]
    plt.scatter([MODEL_SIZE_B], [ep['accv']], marker='*', s=280, color=c, edgecolors='white', linewidths=0.8, label=f"Roberta + LoRA ({ep['name']})")
    plt.text(MODEL_SIZE_B * 1.15, ep['accv'] + (0.004 * (i + 1)), ep['name'], fontsize=9)

plt.xscale('log')
plt.xlabel('Model Size')
plt.ylabel('Accuracy (Vector)')
plt.ylim(y_min, y_max)
plt.xticks([0.1, 1, 10, 100, 1000], ['100M', '1B', '10B', '100B', '1000B'])
plt.title('Figure 2 Baseline + RoBERTa LoRA Experiments (All Completed)')
plt.legend(loc='lower left', fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure2_roberta_experiments_all.png', dpi=150)
plt.show()

print('Completed experiment points:', exp_points)
print('Saved figure:', FIGURES_DIR / 'figure2_roberta_experiments_all.png')

# Table below: just model size vs AccV for completed RoBERTa experiment configs.
if exp_points:
    exp_table = pd.DataFrame([
        {'experiment_name': ep['name'], 'model_size_b': float(MODEL_SIZE_B), 'accv': float(ep['accv'])}
        for ep in exp_points
    ]).sort_values('accv', ascending=False)
    display(exp_table.set_index('experiment_name')[['model_size_b', 'accv']])
else:
    print('No completed experiment points found for table display.')



In [ ]:
# Cell 10: Ensemble top 57c adapters (logit averaging), then clean/evaluate.
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM
from peft import PeftModel

ENSEMBLE_PREFIX = '57c_'
ENSEMBLE_TOP_K = 2
ENSEMBLE_NAME = f'{ENSEMBLE_PREFIX}ensemble_top{ENSEMBLE_TOP_K}'.replace('__', '_')
ENSEMBLE_MODEL_ID = f"{MODEL_ID}_{ENSEMBLE_NAME}"

ENSEMBLE_METRICS_DIR = METRICS_ROOT / ENSEMBLE_NAME
ENSEMBLE_METRICS_DIR.mkdir(parents=True, exist_ok=True)

ENSEMBLE_EVAL_INPUT_PATH = ENSEMBLE_METRICS_DIR / f'{ENSEMBLE_MODEL_ID}_baseline_eval_input.csv'
ENSEMBLE_RAW_OUTPUT_PATH = ENSEMBLE_METRICS_DIR / f'{ENSEMBLE_MODEL_ID}_lora_raw.csv'
ENSEMBLE_CLEAN_OUTPUT_PATH = ENSEMBLE_METRICS_DIR / f'{ENSEMBLE_MODEL_ID}_lora_clean.csv'

if not EXP_SUMMARY_PATH.exists():
    raise FileNotFoundError(f'No experiment summary found at {EXP_SUMMARY_PATH}. Run prior cells first.')

exp_df = pd.read_csv(EXP_SUMMARY_PATH)
if 'experiment_name' not in exp_df.columns or 'accv' not in exp_df.columns:
    raise ValueError('experiment_summary.csv is missing required columns: experiment_name, accv')

exp_df = exp_df.sort_values('timestamp_utc').drop_duplicates('experiment_name', keep='last')
cands = exp_df[
    exp_df['experiment_name'].astype(str).str.startswith(ENSEMBLE_PREFIX)
    & exp_df['accv'].notna()
].copy()

if cands.empty:
    raise RuntimeError(f'No completed experiments found with prefix {ENSEMBLE_PREFIX}.')

cands['accv'] = cands['accv'].astype(float)
selected = cands.sort_values('accv', ascending=False).head(ENSEMBLE_TOP_K).copy()

if len(selected) < ENSEMBLE_TOP_K:
    raise RuntimeError(
        f'Need at least {ENSEMBLE_TOP_K} completed experiments for ensemble, found {len(selected)}.'
    )

print('Selected experiments for ensemble:')
display(selected[['experiment_name', 'accv', 'run_dir']])


def resolve_adapter_path_for_run(run_dir: Path) -> Path:
    summary_path = run_dir / 'training_summary.json'
    if summary_path.exists():
        try:
            obj = json.loads(summary_path.read_text(encoding='utf-8'))
            ckpt = obj.get('best_model_checkpoint')
            if ckpt:
                ckpt_path = Path(str(ckpt))
                if ckpt_path.exists():
                    return ckpt_path
        except Exception as exc:
            print(f'Warning: failed to parse {summary_path}: {exc}')

    state_path = run_dir / 'trainer_state' / 'trainer_state.json'
    if state_path.exists():
        try:
            obj = json.loads(state_path.read_text(encoding='utf-8'))
            ckpt = obj.get('best_model_checkpoint')
            if ckpt:
                ckpt_path = Path(str(ckpt))
                if ckpt_path.exists():
                    return ckpt_path
        except Exception as exc:
            print(f'Warning: failed to parse {state_path}: {exc}')

    return run_dir


def normalize_pred_token(text):
    return re.sub(r'\s+', ' ', str(text)).strip().lower()


eval_source = EXPRESS_ROOT / 'results' / 'roberta-base.csv'
eval_df = pd.read_csv(eval_source)[['segment', 'labels', 'number_of_labels']].copy()
eval_df.to_csv(ENSEMBLE_EVAL_INPUT_PATH, index=False)

if ENSEMBLE_RAW_OUTPUT_PATH.exists():
    ENSEMBLE_RAW_OUTPUT_PATH.unlink()

adapter_paths = []
for _, r in selected.iterrows():
    run_dir = Path(str(r['run_dir']))
    adapter_paths.append(resolve_adapter_path_for_run(run_dir))

print('Resolved adapter paths:')
for ap in adapter_paths:
    print('-', ap)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    token=HF_TOKEN,
    cache_dir=str(MODEL_CACHE_DIR),
    use_fast=True,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
models = []
for ap in adapter_paths:
    base_model = AutoModelForMaskedLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    model = PeftModel.from_pretrained(base_model, str(ap))
    model = model.to(device)
    model.eval()
    models.append(model)

mask_token = tokenizer.mask_token
if mask_token is None:
    raise ValueError('Tokenizer has no mask token; cannot run MLM inference.')


def predict_mask_list_ensemble(segment, num_labels):
    working_text = str(segment)
    target_n = int(num_labels)
    preds = []

    for _ in range(target_n):
        if mask_token not in working_text:
            break

        enc = tokenizer(
            working_text,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_LENGTH,
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        mask_positions = (enc['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=False)
        if len(mask_positions) == 0:
            break

        pos = int(mask_positions[0].item())
        with torch.inference_mode():
            avg_logits = None
            for m in models:
                logits = m(**enc).logits[0, pos]
                avg_logits = logits if avg_logits is None else (avg_logits + logits)
            avg_logits = avg_logits / float(len(models))

        pred_id = int(torch.argmax(avg_logits).item())
        pred_word = normalize_pred_token(tokenizer.decode([pred_id], skip_special_tokens=True))
        if not pred_word:
            pred_word = 'neutral'

        preds.append(pred_word)
        working_text = working_text.replace(mask_token, pred_word, 1)

    if len(preds) != target_n:
        return 'INVALID OUTPUT'
    return str(preds)


outputs = []
for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc=f'RoBERTa LoRA ensemble ({ENSEMBLE_NAME})'):
    outputs.append(predict_mask_list_ensemble(row.segment, row.number_of_labels))

raw_df = eval_df.copy()
raw_df['output'] = outputs
raw_df.to_csv(ENSEMBLE_RAW_OUTPUT_PATH, index=False)
print('Saved ensemble raw outputs to:', ENSEMBLE_RAW_OUTPUT_PATH)

_ = run_cmd(
    [
        'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
        '--input', str(ENSEMBLE_RAW_OUTPUT_PATH),
        '--output', str(ENSEMBLE_CLEAN_OUTPUT_PATH),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

eval_stdout = run_cmd(
    [
        'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
        '--result_path', str(ENSEMBLE_CLEAN_OUTPUT_PATH),
        '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

ensemble_metrics = parse_eval_metrics(eval_stdout)
print('Ensemble metrics:', ensemble_metrics)

# Upsert into shared metrics table.
metrics_row = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'model_id': ENSEMBLE_MODEL_ID,
    'family': MODEL_SPEC['family'],
    'model_name_or_path': MODEL_NAME_OR_PATH,
    'size_b': MODEL_SIZE_B,
    'pipeline': MODEL_SPEC['pipeline'],
    'is_lora': True,
    'v_rate': ensemble_metrics.get('VRate'),
    'accl': ensemble_metrics.get('AccL'),
    'accv': ensemble_metrics.get('AccV'),
    'f1v': ensemble_metrics.get('F1V'),
    'accv2': ensemble_metrics.get('AccV-2'),
    'clean_output_path': str(ENSEMBLE_CLEAN_OUTPUT_PATH),
    'run_dir': ','.join(str(x) for x in adapter_paths),
}

if METRICS_TABLE_PATH.exists():
    metrics_df = pd.read_csv(METRICS_TABLE_PATH)
else:
    metrics_df = pd.DataFrame()

if not metrics_df.empty and {'model_id', 'is_lora'}.issubset(metrics_df.columns):
    metrics_df = metrics_df[~((metrics_df['model_id'] == ENSEMBLE_MODEL_ID) & (metrics_df['is_lora'].astype(str).str.lower() == 'true'))]

metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics_row])], ignore_index=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)
print('Updated metrics table:', METRICS_TABLE_PATH)

# Upsert into experiment summary.
summary_row = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'base_model_id': MODEL_ID,
    'experiment_model_id': ENSEMBLE_MODEL_ID,
    'experiment_name': ENSEMBLE_NAME,
    'train_rows': -1,
    'max_length': MAX_LENGTH,
    'lr': None,
    'epochs': None,
    'train_batch_size': None,
    'grad_accum': None,
    'lora_r': None,
    'lora_alpha': None,
    'lora_dropout': None,
    'target_modules': 'ensemble_logits_avg',
    'v_rate': ensemble_metrics.get('VRate'),
    'accl': ensemble_metrics.get('AccL'),
    'accv': ensemble_metrics.get('AccV'),
    'f1v': ensemble_metrics.get('F1V'),
    'accv2': ensemble_metrics.get('AccV-2'),
    'clean_output_path': str(ENSEMBLE_CLEAN_OUTPUT_PATH),
    'run_dir': ','.join(str(x) for x in adapter_paths),
}

if EXP_SUMMARY_PATH.exists():
    exp_summary_df = pd.read_csv(EXP_SUMMARY_PATH)
else:
    exp_summary_df = pd.DataFrame()

if not exp_summary_df.empty and 'experiment_model_id' in exp_summary_df.columns:
    exp_summary_df = exp_summary_df[exp_summary_df['experiment_model_id'] != ENSEMBLE_MODEL_ID]

exp_summary_df = pd.concat([exp_summary_df, pd.DataFrame([summary_row])], ignore_index=True)
exp_summary_df.to_csv(EXP_SUMMARY_PATH, index=False)
print('Updated experiment summary:', EXP_SUMMARY_PATH)

display(exp_summary_df.sort_values('timestamp_utc').tail(10))

